In [2]:
import numpy as np
import pandas as pd
import optlang
from itertools import repeat
import random
from scipy.stats import qmc
import warnings
import time
import os

warnings.filterwarnings("ignore", category=UserWarning)
from cobra.exceptions import Infeasible
from cobra.io import (load_model, load_json_model, save_json_model, read_sbml_model, write_sbml_model)
from cobra import Model, Reaction, Metabolite, summary

In [6]:
class FluxbalProp:
    def __init__(self,df_fba,rand_seed):
        self.gwth_bound=0.9
        self.flux_max=[-1*x for x in df_fba["FluxMax"].to_numpy()]
        self.rxns_var=[f'EX_{x}' for x in df_fba["SBML ID"].to_numpy()]
        self.lex_rxn=[]
        self.rng=rand_seed
        for x in range(df_fba.shape[0]):
            self.lex_rxn.append('EX_' + df_fba["SBML ID"].iloc[x])
            
def medium_plugin(media_comp,genre): # media_comp being a DataFrame
    medium = {}; mtbl_track=[]
    for ex in genre.exchanges:
        ele=ex.id
        if (media_comp['SBML ID'] == ele[3:]).any() == True:
            mtbl_track.append(genre.metabolites.get_by_id(ele[3:]).name)
            medium.update({ele: (media_comp.loc[media_comp['SBML ID'] == ele[3:], 'mM in media (mmol/L)'].values[0])})
    return medium, mtbl_track

def fluxbal_base(GEM, media, flux_feat, rxn_bound, cons_add, obj_change=None, gene_ko=None): # adjust to help define calculations for gene mutations & other phenomena
    with GEM as model:
        model.medium=media
        if len(cons_add) != 0:
            for c in range(len(cons_add)):
                model.add_cons_vars(cons_add[c])
            
        if obj_change!=None:
            model.objective=obj_change
        if gene_ko != None:
            model.genes.get_by_id(gene_ko).knock_out()
            
        for j in range(len(flux_feat.rxns_var)):
            model.reactions.get_by_id(flux_feat.rxns_var[j]).lower_bound = rxn_bound[j]
            
        model.reactions.PA14_Biomass.bounds = (0.0, flux_feat.gwth_bound)
        try:
            res=model.optimize()
            gwth_output=res.fluxes.PA14_Biomass
            pyoc_output=res.fluxes.DM_cpd01206_c
            
        except Infeasible:
            gwth_output=0.0
            pyoc_output=0.0
            
    return gwth_output, pyoc_output

def fluxbal_slim(GEM, media, flux_feat, rxn_bound): # adjust to help define calculations for gene mutations & other phenomena
    with GEM as model:
        model.medium=media
        model.reactions.PA14_Biomass.upper_bound = flux_feat.gwth_bound
        for j in range(len(flux_feat.rxns_var)):
            model.reactions.get_by_id(flux_feat.rxns_var[j]).lower_bound = rxn_bound[j]
        try:
            res=model.slim_optimize()
            gwth_output=res
        except Infeasible:
            gwth_output=0.0
    del res
    return gwth_output

In [8]:
#---Network reconstruction to GEM---#
# os.chdir("D:/Simul_GEMCM/DataOrg_repository/GEMCM_model")
filename='iPau21_plusAnaerobe.xml'
ipau21=read_sbml_model(filename)
ipau21.add_boundary(ipau21.metabolites.get_by_id("cpd01206_c"),type="demand")
media=pd.read_excel('M9glucose_iPau21.xlsx', sheet_name='Medium_Object')
medium, mtbl_track=medium_plugin(media,ipau21)

#---Simulated medium
flux_info=pd.read_excel('M9glucose_iPau21_exchange_var.xlsx',sheet_name='Set1')
flux_prop=FluxbalProp(flux_info,1229)


['EX_cpd00035_e', 'EX_cpd00051_e', 'EX_cpd00132_e', 'EX_cpd00041_e', 'EX_cpd00023_e', 'EX_cpd00053_e', 'EX_cpd00033_e', 'EX_cpd00119_e', 'EX_cpd00322_e', 'EX_cpd00107_e', 'EX_cpd00039_e', 'EX_cpd00060_e', 'EX_cpd00066_e', 'EX_cpd00129_e', 'EX_cpd00054_e', 'EX_cpd00161_e', 'EX_cpd00069_e', 'EX_cpd00156_e', 'EX_cpd00027_e']
[-5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -10]


### STAGE 1

Hypercube random sampling of lower bound values for selected carbon source exchange reactions

In [3]:
#---------Random sampling of flux boundary values------#
class HypercubeSample:
    def __init__(self,rxns_var,rxns_flux,n_count,rand_seed):
        self.flux_id=rxns_var
        self.flux_base=rxns_flux
        self.sample_scale=[]
        self.N_sample=n_count
        self.rng=rand_seed
        
    def sample_pool(self, lb_val, ub_val):
        for x in range(len(lb_val)):
            self.sample_scale.append([lb_val[x],ub_val[x]])
    
    def lhc_nonshuffle(self,rxns_flux,rxns_var): #new samplings demand new executions of the sample_pool class function
        ub_val=[self.sample_scale[x][1]*rxns_flux[x] for x in range(len(rxns_var))]
        lb_val=[self.sample_scale[x][0]*rxns_flux[x] for x in range(len(rxns_var))]
        lhs_sampler=qmc.LatinHypercube(d=len(rxns_flux),seed=self.rng)
        pool=lhs_sampler.random(n=self.N_sample)
        sample=qmc.scale(pool,ub_val,lb_val)
        df_lhs=pd.DataFrame()
        for w in range(len(rxns_var)):
            df_lhs[rxns_var[w]]=sample[:,w]
        return df_lhs

### Batch sampling

"LowGrowth":
- n = 4000 with [0, (0.3 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange

"HighGrowth":
- n = 1200 with [0, (1 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange
- n = 800 with [0, (0.7 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange
- n = 800 with [0, (0.5 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange
- n = 600 with [0, (0.32 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange
- n = 600 with [0, (0.22 x 5)] amino acid exchanges and [0, (1 x 10)] glucose exchange
- n = 600 with [0, (0.18 x 5)] amino acid exchanges and [0, (0.5 x 10)] glucose exchange
- n = 800 with [0, (0.1 x 5)] amino acid exchanges and [0, (0.1 x 10)] glucose exchange
- n = 600 with [0, (0.14 x 5)] amino acid exchanges and [0, (0.25 x 10)] glucose exchange

In [12]:
g=HypercubeSample(flux_prop.rxns_var,flux_prop.flux_max,2000,1229)
#---*_val boundaries experessed as percentile of maximum flux bound
ub_val=[0.5 for x in range(len(flux_prop.rxns_var)-1)] # amino acid excanges
ub_val.append(1.0) # glucose exchange
lb_val=[0.1 for x in range(len(flux_prop.rxns_var)-1)] # amino acid excanges
lb_val.append(0.5) 

g.sample_pool(lb_val, ub_val)
df_lhs=g.lhc_nonshuffle(flux_prop.flux_max, flux_prop.rxns_var)

In [13]:
df_lhs.to_csv("ExchangeBounds_Training_LOWGROWTH.csv", index=False)
# df_lhs.to_csv('ExchangeBounds_Training_LOWGROWTH.csv', mode='a', index=False, header=False) #for appending batch hypercube sampling sets

### Stage 2: FBA training data generation

HIGHGROWTH exchange flux constraints used to prepare ensemble model for GEM-CM

LOWGROWTH exchange flux constraints used to evaluate carbon source exchange reactions most important to regression

***FBA training data from HIGHGROWTH constraints labeled FinalEnsem***

***FBA training data from LOWGROWTH constraints labeled PredictorREDUCE***

In [ ]:
file_path1='C:/Users/msk4u/Ford Lab/FBA_neuralnet/'
dataset_tot=pd.read_csv(file_path1+'ExchangeBounds_Training_HIGHGROWTH.csv')
flux_ranges=dataset_tot.to_numpy()

# Chunking exchange flux set for FBA simulation
run_set=[]
run_ind=[int(x) for x in np.linspace(0,len(flux_ranges),27)]
for j in range(1,len(run_ind)):
    run_set.append([run_ind[j-1],run_ind[j]])
print(run_set)

In [ ]:
#---FBA objective for pyocyanin synthesis
obj_test2=ipau21.problem.Objective(ipau21.reactions.PA14_Biomass.flux_expression + ipau21.reactions.DM_cpd01206_c.flux_expression, 
                                   direction='max')
#---Additional constraints may be defined, if desired
cons_add=[]

In [ ]:
gwth_step=0.2 #variable 'Gamma' in objective decision tree
ko_id = None #"PA14_09490 (phzM),PA14_64200 (purH), PA14_18740 (argG), PA14_40670 (metH)
for k in range(len(run_set)):
    col2=[x for x in flux_prop.rxns_var]
    fba_output=[]
    biom_output=[]; pyoc_output=[]
    print(f"Starting set {k+1}")
    for n in range(run_set[k][0],run_set[k][1]):
        if n%50 == 0:
            print(f"Fluxes {n+1} out of {len(flux_ranges)} completed")
            time.sleep(1)
        fba_output.append(list(flux_ranges[n]))
        res_biom, res_pyoc=fluxbal_base(ipau21, medium, flux_prop, list(flux_ranges[n]),
                                        cons_add, obj_change=None, gene_ko=ko_id)
        #Testing condition if it meets criterion for simultaneous growth and pyocyanin synthesis
        if 0.05 <= res_biom < gwth_step:
#         if res_biom >= 0.05:
            res2_biom, res2_pyoc=fluxbal_base(ipau21, medium, flux_prop, list(flux_ranges[n]),
                                              cons_add, obj_change=obj_test2, gene_ko=ko_id)
            if res2_biom > 0.05:
                biom_output.append(res2_biom)
                pyoc_output.append(res2_pyoc)
                
            else:
                biom_output.append(res_biom)
                pyoc_output.append(res_pyoc)
            
        else:
            biom_output.append(res_biom)
            pyoc_output.append(res_pyoc)
            
    df_output=pd.DataFrame(data=fba_output,columns=col2)
    biom_lbcheck=np.zeros(len(biom_output))
    pyoc_lbcheck=np.zeros(len(pyoc_output))
    for i in range(len(biom_output)):
        if biom_output[i] > 0:
            biom_lbcheck[i]=biom_output[i]
        else:
            continue
        
    df_output["PA14_Biomass"]=biom_lbcheck
    df_output["DM_cpd01206_c"]=pyoc_output

    if k == 0:
        df_output.to_csv("FBACalc_ObjC2_Step02_FinalEnsem_del purH.csv",index=False)
        print(f"Added set {k+1}")
    else:
        df_output.to_csv("FBACalc_ObjC2_Step02_FinalEnsem_del purH.csv", mode='a', index=False, header=False)
        print(f'Appended set {k+1}')
    print('Moving to next set...')
    print()